# Test Postgre

## Init

In [1]:
# ========== 初始化: 自动检测 PostgreSQL 路径（版本匹配） ==========
import subprocess, time, shutil, os, glob
from pathlib import Path

# 自动检测 PostgreSQL 路径
PSQL = shutil.which("psql")
if not PSQL:
    raise FileNotFoundError("找不到 psql，请安装 PostgreSQL 客户端")

def _get_pg_version(data_dir: str) -> str:
    """读取数据目录的 PG 主版本号。"""
    vf = os.path.join(data_dir, "PG_VERSION")
    if os.path.isfile(vf):
        with open(vf) as f:
            return f.read().strip()
    return None

def _find_pg_binary(name: str, data_dir: str = None) -> str:
    """查找与数据目录版本匹配的 pg 工具。"""
    # 1. 从数据目录版本推断优先搜索路径
    search_dirs = []
    if data_dir:
        ver = _get_pg_version(data_dir)
        if ver:
            # 搜索包含版本号的路径: pgsql-13.1, postgresql-13, etc.
            patterns = [
                f"/home/*/pgsql-{ver}.*/bin",
                f"/usr/local/pgsql/{ver}*/bin",
                f"/usr/local/pgsql-{ver}*/bin",
                f"/usr/lib/postgresql/{ver}/bin",
            ]
            for pat in patterns:
                for d in sorted(glob.glob(pat)):
                    if os.path.isdir(d):
                        search_dirs.append(d)
    # 2. shutil.which 找到的（验证版本兼容性）
    found = shutil.which(name)
    if found:
        search_dirs.append(str(Path(found).parent))
    # 3. 额外路径
    for d in ["/usr/local/pgsql/13.1/bin", "/usr/local/pgsql/16/bin", "/usr/lib/postgresql/16/bin"]:
        if os.path.isdir(d):
            search_dirs.append(d)
    for d in search_dirs:
        candidate = os.path.join(d, name)
        if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    raise FileNotFoundError(f"找不到与 PG 数据目录版本匹配的 {name}")

# 从运行中的 PG 获取 data_directory
r = subprocess.run(
    [PSQL, "-U", "postgres", "-t", "-A", "-c", "SHOW data_directory;"],
    capture_output=True, text=True
)
if r.returncode == 0:
    PG_DATA = r.stdout.strip()
else:
    PG_DATA = os.environ.get("PGDATA", "/home/liwei/pgdata")

PG_CTL = _find_pg_binary("pg_ctl", PG_DATA)

print(f"PSQL:     {PSQL}")
print(f"PG_CTL:   {PG_CTL}")
print(f"PG_DATA:  {PG_DATA}")
if PG_DATA:
    print(f"PG 版本:  {_get_pg_version(PG_DATA)}")


def cold_restart_pg():
    """冷重启 PG：停 PG → 清 OS page cache → 启 PG（使用版本匹配的 pg_ctl）。"""
    t0 = time.perf_counter()
    # 1. 停 PG
    subprocess.run([PG_CTL, "-D", PG_DATA, "-m", "fast", "stop"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # 2. 清 OS page cache
    try:
        with open("/proc/sys/vm/drop_caches", "w") as f:
            f.write("3\n")
    except PermissionError:
        pass
    # 3. 启 PG（版本匹配，不会失败）
    subprocess.run([PG_CTL, "-D", PG_DATA, "start"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # 4. 等待 ready
    for i in range(120):
        r = subprocess.run(
            [PSQL, "-U", "postgres", "-t", "-A", "-c", "SELECT 1;"],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            elapsed = time.perf_counter() - t0
            print(f"  冷重启完成 ({elapsed:.1f}s, 等待 {i}s)")
            return elapsed
        time.sleep(1)
    raise RuntimeError("PostgreSQL 启动超时（120 秒未 ready）")


print("初始化完成\n")


PSQL:     /home/liwei/miniconda3/envs/TestEnv/bin/psql
PG_CTL:   /home/liwei/pgsql-13.1/bin/pg_ctl
PG_DATA:  /home/liwei/pgdata
PG 版本:  13
初始化完成



In [2]:
import os
import sys
import re
import time
import subprocess
import shlex
import csv
from pathlib import Path

# 自动检测 psql 路径（优先使用第一个 cell 中已检测的 PSQL 变量）
try:
    PSQL_BIN = PSQL
except NameError:
    import shutil
    PSQL_BIN = shutil.which("psql")
    if not PSQL_BIN:
        PSQL_BIN = "/usr/local/pgsql/13.1/bin/psql"  # fallback

# TO CHANGE: 统计信息收集并行度
PG_PARALLELISM = 8

# 添加 scripts 目录到路径
sys.path.append(os.path.abspath('../scripts'))

from extract_card_from_pg_plan import extract_cardinalities

project_root = Path('..').resolve()
benchmark_dir = project_root / "Benchmark" / "workloads"

running_space = Path("./running_space").resolve()
running_space.mkdir(parents=True, exist_ok=True)

checkpoint_dir = Path("./checkpoint/Postgre").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# PostgreSQL 连接信息（可通过环境变量覆盖）
PG_CONN_BASE = os.environ.get(
    "PG_CONN_BASE",
    f"host=127.0.0.1 port=5432 user={os.environ.get('USER', 'postgres')}"
)

def make_conn_str(db_name: str) -> str:
    return f"{PG_CONN_BASE} dbname={db_name}"


BENCHMARKS = {
    "STATS": {
        "db_name": "stats",
        "queries_file": benchmark_dir / "STATS-CEB" / "queries.sql",
        "subquery_file": benchmark_dir / "STATS-CEB" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_stats.txt",
        "explain_output": running_space / "pg_stats_explain.txt",
        "explain_queries_output": running_space / "pg_stats_queries_explain.txt",
    },
    "JOBLight": {
        "db_name": "imdblight",
        "queries_file": benchmark_dir / "JOBLight" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBLight" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_joblight.txt",
        "explain_output": running_space / "pg_joblight_explain.txt",
        "explain_queries_output": running_space / "pg_joblight_queries_explain.txt",
    },
    "JOBLightRanges": {
        "db_name": "imdblightranges",
        "queries_file": benchmark_dir / "JOBLightRanges" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBLightRanges" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_joblr.txt",
        "explain_output": running_space / "pg_joblr_explain.txt",
        "explain_queries_output": running_space / "pg_joblr_queries_explain.txt",
    },
    "JOBM": {
        "db_name": "imdbm",
        "queries_file": benchmark_dir / "JOBM" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBM" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_jobm.txt",
        "explain_output": running_space / "pg_jobm_explain.txt",
        "explain_queries_output": running_space / "pg_jobm_queries_explain.txt",
    },
    "StatsJoin": {
        "db_name": "stats",
        "queries_file": benchmark_dir / "StatsJoin" / "queries.sql",
        "subquery_file": benchmark_dir / "StatsJoin" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_statsjoin.txt",
        "explain_output": running_space / "pg_statsjoin_explain.txt",
        "explain_queries_output": running_space / "pg_statsjoin_queries_explain.txt",
    },
}

print("配置完成，可以开始收集统计信息与基数估计结果")

配置完成，可以开始收集统计信息与基数估计结果


In [3]:
def prepare_explain_sql(queries_file: Path, output_file: Path) -> None:
    """
    复制 queries 文件到 output_file，并在每个 SELECT 语句前添加 EXPLAIN。
    """
    if not queries_file.exists():
        raise FileNotFoundError(f"找不到 queries 文件: {queries_file}")

    content = queries_file.read_text(encoding="utf-8")
    content = re.sub(r"count\s*\(\s*\*\s*\)", "*", content, flags=re.IGNORECASE)
    pattern = r"^(\s*)(select\s+)"
    replacement = r"\1EXPLAIN \2"
    new_content = re.sub(pattern, replacement, content, flags=re.IGNORECASE | re.MULTILINE)

    output_file.write_text(new_content, encoding="utf-8")


def run_psql_sql(conn_str: str, sql: str) -> str:
    cmd = [
        PSQL_BIN,
        "-v",
        "ON_ERROR_STOP=1",
        conn_str,
        "-t",
        "-A",
        "-c",
        sql
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "psql 执行失败")
    return result.stdout


def run_psql_file(conn_str: str, sql_file: Path, output_file: Path) -> None:
    cmd = [
        PSQL_BIN,
        "-v",
        "ON_ERROR_STOP=1",
        conn_str,
        "-f",
        str(sql_file),
        "-o",
        str(output_file)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "psql 执行失败")


def _get_user_tables(conn_str: str):
    """获取所有用户表（跳过 _ss1d0 采样表），返回 [(schema, table), ...]。"""
    sql = (
        "SELECT schemaname, tablename FROM pg_tables "
        "WHERE schemaname NOT IN ('pg_catalog', 'information_schema') "
        "  AND strpos(tablename, '_ss1d0') = 0"
    )
    output = run_psql_sql(conn_str, sql)
    tables = []
    for line in output.strip().splitlines():
        if not line.strip():
            continue
        parts = line.strip().split('|')
        if len(parts) >= 2:
            tables.append((parts[0].strip(), parts[1].strip()))
    return tables


def _find_unindexed_column(conn_str: str, schema: str, table: str) -> str:
    """找到表上没有索引的列，用于 ORDER BY 触发全表扫描预热。"""
    # 获取所有列
    cols_sql = (
        f"SELECT a.attname FROM pg_attribute a "
        f"JOIN pg_class c ON a.attrelid = c.oid "
        f"JOIN pg_namespace n ON c.relnamespace = n.oid "
        f"WHERE n.nspname = '{schema}' AND c.relname = '{table}' "
        f"  AND a.attnum > 0 AND NOT a.attisdropped "
        f"ORDER BY a.attnum"
    )
    all_cols = run_psql_sql(conn_str, cols_sql).strip().splitlines()
    if not all_cols:
        return "id"

    # 获取索引列
    idx_sql = (
        f"SELECT indexdef FROM pg_indexes "
        f"WHERE schemaname = '{schema}' AND tablename = '{table}'"
    )
    idx_defs = run_psql_sql(conn_str, idx_sql).strip().splitlines()
    indexed_cols = set()
    for idx_def in idx_defs:
        m = re.search(r"\(([^)]+)\)", idx_def)
        if m:
            for col in m.group(1).split(','):
                indexed_cols.add(col.strip())

    for col in all_cols:
        if col not in indexed_cols:
            return col
    # fallback: 最后一列
    return all_cols[-1]


def collect_pg_statistics(conn_str: str, run_vacuum: bool = False):
    """
    收集 PostgreSQL 统计信息并返回耗时（秒），热启动口径。
    流程：冷重启 PG → 逐表 SELECT * ORDER BY 预热 → 逐表 ANALYZE 计时。
    """
    # 1. 冷启动
    cold_restart_pg()

    # 2. 获取用户表
    tables = _get_user_tables(conn_str)
    print(f"  共 {len(tables)} 张表待 ANALYZE")

    # 3. 逐表：预热（不计时）→ ANALYZE（计时）
    total_elapsed = 0.0
    for schema, table in tables:
        # 找到无索引列
        col = _find_unindexed_column(conn_str, schema, table)
        # 预热：SELECT * ORDER BY col LIMIT 5 触发全表扫描，加载数据到 OS cache
        run_psql_sql(
            conn_str,
            f'SELECT * FROM "{schema}"."{table}" ORDER BY "{col}" LIMIT 5'
        )
        # 计时 ANALYZE
        start = time.perf_counter()
        run_psql_sql(conn_str, f'ANALYZE "{schema}"."{table}"')
        elapsed = time.perf_counter() - start
        total_elapsed += elapsed
        print(f"    {schema}.{table}: {elapsed:.3f}s (prewarmed via ORDER BY {col})")

    print(f"  逐表 ANALYZE 合计: {total_elapsed:.3f}s")

    if run_vacuum:
        run_psql_sql(conn_str, "VACUUM FULL pg_statistic;")
        run_psql_sql(conn_str, "VACUUM FULL pg_statistic_ext_data;")
    return total_elapsed


def get_pg_statistics_size(conn_str: str):
    """获取 PostgreSQL 统计信息大小（字节）。"""
    size_query = (
        "SELECT pg_total_relation_size('pg_statistic') "
        "+ pg_total_relation_size('pg_statistic_ext_data');"
    )
    try:
        output = run_psql_sql(conn_str, size_query)
        return int(output.strip().splitlines()[-1])
    except Exception:
        try:
            output = run_psql_sql(conn_str, "SELECT pg_total_relation_size('pg_statistic');")
            return int(output.strip().splitlines()[-1])
        except Exception as e:
            raise RuntimeError(f"统计信息大小获取失败: {e}") from e


def extract_pg_cardinalities(conn_str: str, subquery_file: Path, explain_output: Path):
    """对子查询文件生成 EXPLAIN SQL，运行 psql，提取基数估计。"""
    explain_sql = running_space / "explain.sql"
    prepare_explain_sql(subquery_file, explain_sql)
    run_psql_file(conn_str, explain_sql, explain_output)
    return extract_cardinalities(explain_output)


def time_explain_queries(conn_str: str, queries_file: Path, explain_output: Path) -> float:
    """对主查询文件生成 EXPLAIN SQL，运行 psql，返回 wall-clock 耗时（秒）。"""
    explain_sql = running_space / "explain_queries.sql"
    prepare_explain_sql(queries_file, explain_sql)
    start = time.perf_counter()
    run_psql_file(conn_str, explain_sql, explain_output)
    return time.perf_counter() - start


In [4]:
def evaluate_pg_benchmark(benchmark_name: str, collect_stats: bool = True, run_vacuum: bool = False):
    if benchmark_name not in BENCHMARKS:
        raise ValueError(f"不支持的 benchmark: {benchmark_name}")

    cfg = BENCHMARKS[benchmark_name]
    conn_str = make_conn_str(cfg["db_name"])

    print(f"\n{'=' * 60}")
    print(f"Benchmark: {benchmark_name}")
    print(f"数据库: {cfg['db_name']}")
    if cfg["queries_file"] is not None:
        print(f"主查询文件: {cfg['queries_file']}")
    if cfg["subquery_file"] is not None:
        print(f"子查询文件: {cfg['subquery_file']}")

    stats_time = None
    stats_size = None
    if collect_stats:
        stats_time = collect_pg_statistics(conn_str, run_vacuum=run_vacuum)
    stats_size = get_pg_statistics_size(conn_str)

    cardinals = None
    if cfg["subquery_file"] is not None:
        cardinals = extract_pg_cardinalities(
            conn_str,
            cfg["subquery_file"],
            cfg["explain_output"]
        )
        cfg["card_output"].write_text("\n".join(str(c) for c in cardinals), encoding="utf-8")

    eval_time = None
    if cfg["queries_file"] is not None:
        eval_time = time_explain_queries(
            conn_str,
            cfg["queries_file"],
            cfg["explain_queries_output"]
        )

    print("\n结果:")
    print(f"  统计信息收集时间 (热启动, 逐表预热): {stats_time} 秒")
    print(f"  统计信息大小: {stats_size} 字节")
    if eval_time is not None:
        print(f"  主查询 EXPLAIN 时间: {eval_time} 秒")
    if cardinals is not None:
        print(f"  基数数量: {len(cardinals)}")

    return {
        "stats_time": stats_time,
        "stats_size": stats_size,
        "eval_time": eval_time,
        "cardinalities": cardinals
    }

In [5]:
results = {}
for name in BENCHMARKS:
    results[name] = evaluate_pg_benchmark(name, collect_stats=True, run_vacuum=False)


Benchmark: STATS
数据库: stats
主查询文件: /home/liwei/starCE/Benchmark/workloads/STATS-CEB/queries.sql
子查询文件: /home/liwei/starCE/Benchmark/workloads/STATS-CEB/subquery/subquery.sql
  冷重启完成 (0.7s, 等待 0s)
  共 8 张表待 ANALYZE
    public.users: 0.132s (prewarmed via ORDER BY reputation)
    public.posts: 0.146s (prewarmed via ORDER BY posttypeid)
    public.postlinks: 0.038s (prewarmed via ORDER BY creationdate)
    public.posthistory: 0.129s (prewarmed via ORDER BY posthistorytypeid)
    public.comments: 0.114s (prewarmed via ORDER BY score)
    public.votes: 0.123s (prewarmed via ORDER BY votetypeid)
    public.badges: 0.069s (prewarmed via ORDER BY date)
    public.tags: 0.024s (prewarmed via ORDER BY count)
  逐表 ANALYZE 合计: 0.775s

结果:
  统计信息收集时间 (热启动, 逐表预热): 0.7749850302934647 秒
  统计信息大小: 1220608 字节
  主查询 EXPLAIN 时间: 0.7439129143022001 秒
  基数数量: 2471

Benchmark: JOBLight
数据库: imdblight
主查询文件: /home/liwei/starCE/Benchmark/workloads/JOBLight/queries.sql
子查询文件: /home/liwei/starCE/Benchmark/workl

In [6]:
stats_summary = {
    name: {
        "stats_time": results[name]["stats_time"],
        "stats_size": results[name]["stats_size"],
        "eval_time": results[name]["eval_time"],
    }
    for name in results
}

stats_summary_path = checkpoint_dir / "pg_stats_summary.csv"
with stats_summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Benchmark", "BuildTime", "StatisticsSize", "EvaluationTime"])
    for name, info in stats_summary.items():
        writer.writerow([name, info["stats_time"], info["stats_size"], info["eval_time"] or ""])
print(f"统计信息已保存: {stats_summary_path}")

统计信息已保存: /home/liwei/starCE/experiment/checkpoint/Postgre/pg_stats_summary.csv
